# RR SSC-MEK PHASE A: survey only (NO SCORING)
Question under design: does the audited L1000 pipeline, queried with systemic sclerosis skin data that played no part in the original MEK finding, place MEK inhibitors among the reversers?

This notebook computes **no connectivity score and no disease signature**. It reads metadata only: which LINCS compounds are MEK inhibitors and in which cell lines they were profiled, and what each candidate SSc skin series actually deposits. The pre-registration is written from this report and committed before any score exists.

P01's source data (GSE138669, and GSE195452 for the upstream analysis) are excluded by design and are not fetched.

Run: hamburger (top left) > Runtime > Run all. Then copy the text between `=== SSC-MEK PHASE A REPORT BEGIN ===` and `=== SSC-MEK PHASE A REPORT END ===` and paste it back.

In [ ]:
# CELL 1: mount Drive, work folder (new folder; the benchmark folder is read, never written)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
BENCH = "/content/drive/MyDrive/RitschelResearch/benchmark"
W = "/content/drive/MyDrive/RitschelResearch/ssc_mek"; os.makedirs(W, exist_ok=True); os.chdir(W)
print("workdir", os.getcwd()); print("benchmark folder present:", os.path.isdir(BENCH))

In [ ]:
# CELL 2: LINCS 2020 metadata. Copied from the benchmark folder when present, otherwise downloaded.
import urllib.request, os, shutil, pandas as pd
B = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/"
META = {}
for f in ["siginfo_beta.txt", "compoundinfo_beta.txt", "geneinfo_beta.txt", "cellinfo_beta.txt"]:
    if not os.path.exists(f):
        src = os.path.join(BENCH, f)
        if os.path.exists(src): shutil.copy(src, f)
        else: urllib.request.urlretrieve(B + f, f)
    META[f] = os.path.getsize(f); print(f, META[f])
g = pd.read_csv("geneinfo_beta.txt", sep="\t")
LM = g.loc[g.feature_space == "landmark", ["gene_id", "gene_symbol"]].copy(); LM["gene_id"] = LM.gene_id.astype(int)
assert len(LM) == 978, "LANDMARK COUNT IS NOT 978"
print("landmark genes:", len(LM))

In [ ]:
# CELL 3: the library as the benchmark defined it (QC-pass exemplar trt_cp), broken down by cell line.
# Prints what cellinfo actually says about each line. No mesenchymal rule is applied here: the rule is
# written from this table, not from what the labels ought to say.
import pandas as pd
si = pd.read_csv("siginfo_beta.txt", sep="\t", usecols=["sig_id", "pert_id", "pert_type", "is_exemplar_sig", "qc_pass", "cell_iname"], low_memory=False)
si = si[(si.pert_type == "trt_cp") & (si.is_exemplar_sig == 1) & (si.qc_pass == 1)].copy()
LIB = dict(sigs=int(len(si)), compounds=int(si.pert_id.nunique()), cells=int(si.cell_iname.nunique()))
print("LIBRARY", LIB)
ce = pd.read_csv("cellinfo_beta.txt", sep="\t", dtype=str).fillna("")
print("cellinfo columns:", list(ce.columns))
keep = [c for c in ["cell_iname", "cell_type", "cell_lineage", "primary_disease", "subtype", "growth_pattern"] if c in ce.columns]
ct = si.groupby("cell_iname").agg(n_sigs=("sig_id", "size"), n_cmpds=("pert_id", "nunique")).reset_index()
CT = ct.merge(ce[keep].drop_duplicates("cell_iname"), on="cell_iname", how="left").fillna("?").sort_values("n_sigs", ascending=False)
assert int(CT.n_sigs.sum()) == LIB["sigs"], "CELL TABLE DOES NOT SUM TO THE LIBRARY"
print(CT.to_string(index=False))

In [ ]:
# CELL 4: compounds. The MEK class is read from the moa and target fields, and every moa string that
# mentions MEK, ERK or MAPK is printed first so that nothing hides in the vocabulary.
import pandas as pd, re, collections
ci = pd.read_csv("compoundinfo_beta.txt", sep="\t", dtype=str).fillna("")
print("compoundinfo columns:", list(ci.columns))
voc = ci[ci.moa.str.contains(r"MEK|ERK|MAPK|MAP kinase", case=False, regex=True)]
VOC = voc.groupby("moa").pert_id.nunique().sort_values(ascending=False)
print("--- moa strings mentioning MEK / ERK / MAPK, with compound counts ---"); print(VOC.to_string())
sigs_by = si.groupby("pert_id")
def cov(pid):
    if pid not in sigs_by.groups: return 0, 0, ""
    d = sigs_by.get_group(pid); vc = d.cell_iname.value_counts()
    return int(len(d)), int(vc.size), " ".join(f"{k}:{v}" for k, v in vc.items())
def describe(sub, why):
    rows = []
    for pid, d in sub.groupby("pert_id"):
        n, nc, cells = cov(pid)
        rows.append(dict(why=why, pert_id=pid, cmap_name="|".join(sorted(set(d.cmap_name))), moa="|".join(sorted(set(x for x in d.moa if x))),
                         target="|".join(sorted(set(x for x in d.target if x))), inchi_key=sorted(set(d.inchi_key))[0],
                         aliases="|".join(sorted(set(x for x in d.compound_aliases if x)))[:60], n_lib_sigs=n, n_cells=nc, cells=cells))
    return rows
is_mek = ci.moa.str.contains(r"\bMEK inhibitor", case=False, regex=True) | ci.target.isin(["MAP2K1", "MAP2K2"])
mek_ids = set(ci[is_mek].pert_id)
ROWS = describe(ci[ci.pert_id.isin(mek_ids)], "MEK class (moa or target)")
def by_name(names):
    n = [x.lower() for x in names]
    al = ci.compound_aliases.str.lower().str.split("|").apply(lambda L: any(x.strip() in n for y in L for x in y.split(",")))
    return ci[ci.cmap_name.str.lower().isin(n) | al]
NAMED = {"PD-0325901 (the named compound)": ["pd-0325901", "pd0325901", "pd 0325901", "mirdametinib", "pd-325901"],
         "reference: dasatinib": ["dasatinib"], "reference: bosutinib": ["bosutinib"], "reference: nintedanib": ["nintedanib", "bibf-1120", "bibf1120"],
         "reference: mycophenolate": ["mycophenolic-acid", "mycophenolic acid", "mycophenolate-mofetil", "mycophenolate mofetil"]}
for why, names in NAMED.items():
    sub = by_name(names); pids = set(sub.pert_id)
    if not pids: ROWS.append(dict(why=why, pert_id="NOT FOUND BY NAME OR ALIAS", cmap_name="", moa="", target="", inchi_key="", aliases="", n_lib_sigs=0, n_cells=0, cells=""))
    else: ROWS += describe(ci[ci.pert_id.isin(pids)], why)
D = pd.DataFrame(ROWS).sort_values(["why", "n_lib_sigs"], ascending=[True, False])
D.to_csv("sscmek_compounds_phaseA.csv", index=False)
back = pd.read_csv("sscmek_compounds_phaseA.csv", dtype=str)          # a write is not done until a read confirms it
assert len(back) == len(D), "COMPOUND TABLE READ-BACK MISMATCH"
print("--- compounds ---"); print(D[["why", "pert_id", "cmap_name", "moa", "target", "inchi_key", "n_lib_sigs", "n_cells"]].to_string(index=False))
print("--- per-compound cell lines (cell:signatures), compounds with at least one library signature ---")
for _, r in D[D.n_lib_sigs > 0].iterrows(): print(f"{r.cmap_name} [{r.pert_id}] {r.cells}")

In [ ]:
# CELL 5: what each candidate SSc skin series actually deposits. Metadata only.
# The candidate list is from recall and is NOT trusted: every line below is read from the deposit.
# Variables are found by FIELD NAME across all characteristics lines, never by line number.
import gzip, urllib.request, os, re, collections, io
CAND = ["GSE58095", "GSE130955", "GSE95065", "GSE181549", "GSE125362", "GSE76886", "GSE9285", "GSE32413", "GSE45485", "GSE59785"]
EXCLUDED_BY_DESIGN = ["GSE138669", "GSE195452"]      # P01's source data; never fetched here
UA = {"User-Agent": "Mozilla/5.0 (research survey; contact glen@ritschelresearch.com)"}
def fetch(u, timeout=180):
    return urllib.request.urlopen(urllib.request.Request(u, headers=UA), timeout=timeout).read()
def matrix_files(g):
    base = "https://ftp.ncbi.nlm.nih.gov/geo/series/%snnn/%s/matrix/" % (g[:-3], g)
    html = fetch(base, 60).decode("utf-8", "replace")
    names = sorted(set(re.findall(r'href="([^"/]+_series_matrix\.txt\.gz)"', html)))
    return base, names
def strip(x): return x.strip().strip('"').strip()
def parse_matrix(path):
    hdr = collections.defaultdict(list); nrow = 0; inb = False; first = None; vals = []
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("!series_matrix_table_end"): break
            if inb:
                nrow += 1
                if nrow == 1: continue                      # header row of the table
                q = line.rstrip("\n").split("\t")
                if first is None: first = strip(q[0])
                if nrow <= 3001:
                    for x in q[1:21]:
                        try: vals.append(float(strip(x)))
                        except ValueError: pass
                continue
            if line.startswith("!series_matrix_table_begin"): inb = True; continue
            if line.startswith("!"):
                k, _, rest = line.rstrip("\n").partition("\t")
                hdr[k].append([strip(x) for x in rest.split("\t")])
    S = lambda k: [x for row in hdr.get(k, []) for x in row]
    n = len(S("!Sample_geo_accession"))
    fields = collections.OrderedDict()
    for row in hdr.get("!Sample_characteristics_ch1", []):
        for v in row:
            if not v: continue
            name, sep, val = v.partition(":")
            name = name.strip() if sep else "(no field name)"; val = val.strip() if sep else v
            fields.setdefault(name, collections.Counter())[val] += 1
    vals.sort()
    pct = lambda p: (vals[min(len(vals) - 1, int(p * len(vals)))] if vals else None)
    return dict(n=n, title=" ".join(S("!Series_title")), summary=" ".join(S("!Series_summary"))[:420], design=" ".join(S("!Series_overall_design"))[:420],
                submitted=" ".join(S("!Series_submission_date")), pubmed=",".join(S("!Series_pubmed_id")), platform=S("!Series_platform_id"),
                contributors=S("!Series_contributor")[:4], series_supp=S("!Series_supplementary_file"),
                fields=fields, source=collections.Counter(S("!Sample_source_name_ch1")), titles=S("!Sample_title")[:6],
                channels=collections.Counter(S("!Sample_channel_count")), strategy=collections.Counter(S("!Sample_library_strategy")),
                sample_supp_none=sum(1 for k in hdr if k.startswith("!Sample_supplementary_file") for row in hdr[k] for x in row if x.upper() == "NONE"),
                probes=max(nrow - 1, 0), first=first, p01=pct(0.01), p50=pct(0.50), p99=pct(0.99))
def landmark_cov(plat, lmids):
    f = plat + ".annot.gz"
    if not os.path.exists(f):
        try: open(f, "wb").write(fetch("https://ftp.ncbi.nlm.nih.gov/geo/platforms/%snnn/%s/annot/%s.annot.gz" % (plat[:-3], plat, plat)))
        except Exception as e: return "annot unavailable (%s)" % type(e).__name__
    ids = set(); inb = False; ige = None
    for raw in gzip.open(f, "rt", encoding="utf-8", errors="replace"):
        if raw.startswith("!platform_table_end"): break
        if inb:
            q = raw.rstrip("\n").split("\t")
            if ige is None: ige = [i for i, c in enumerate(q) if c.strip() == "Gene ID"]; continue
            if ige and len(q) > ige[0]:
                x = q[ige[0]].split("///")[0].strip()
                if x.isdigit(): ids.add(int(x))
        if raw.startswith("!platform_table_begin"): inb = True
    return "%d of %d landmarks" % (len(ids & lmids), len(lmids))
def top(counter, k=8): return "; ".join(f"{a!r}:{b}" for a, b in counter.most_common(k)) + (" ..." if len(counter) > k else "")
def survey(g, lmids, get_files=matrix_files, get=fetch):
    out = []
    try: base, names = get_files(g)
    except Exception as e: return [f"{g}: MATRIX LISTING FAILED ({type(e).__name__}: {e})"]
    if not names: return [f"{g}: NO SERIES MATRIX FILES LISTED"]
    out.append(f"{g}: {len(names)} matrix file(s): {names}")
    for nm in names:
        if not os.path.exists(nm):
            try: open(nm, "wb").write(get(base + nm))
            except Exception as e: out.append(f"  {nm}: FETCH FAILED ({type(e).__name__}: {e})"); continue
        try: m = parse_matrix(nm)
        except Exception as e: out.append(f"  {nm}: PARSE FAILED ({type(e).__name__}: {e})"); continue
        out.append(f"  [{nm}] samples {m['n']} | platform {m['platform']} | submitted {m['submitted']} | pubmed {m['pubmed'] or '-'} | channels {dict(m['channels'])} | strategy {dict(m['strategy']) or '-'}")
        out.append(f"    title: {m['title']}")
        out.append(f"    contributors: {m['contributors']}")
        out.append(f"    summary: {m['summary']}")
        out.append(f"    design: {m['design']}")
        out.append(f"    table: {m['probes']} rows, first id {m['first']!r}, values p01 {m['p01']} p50 {m['p50']} p99 {m['p99']} | samples with supplementary NONE: {m['sample_supp_none']}")
        out.append(f"    series supplementary: {' | '.join(x.split('/')[-1] for x in m['series_supp']) or 'NONE'}")
        out.append(f"    source_name: {top(m['source'])}")
        out.append(f"    sample titles: {m['titles']}")
        for name, c in m["fields"].items():
            out.append(f"    field {name!r} ({sum(c.values())} of {m['n']} samples, {len(c)} values): {top(c)}")
        if m["probes"] > 0:
            for p in m["platform"]: out.append(f"    landmark coverage {p}: {landmark_cov(p, lmids)}")
    return out
lmids = set(LM.gene_id); REP = []
for g in CAND:
    assert g not in EXCLUDED_BY_DESIGN
    REP += survey(g, lmids); print(REP[-1][:120] if REP else g, flush=True)
open("sscmek_series_survey_phaseA.txt", "w", encoding="utf-8").write("\n".join(REP))
assert open("sscmek_series_survey_phaseA.txt", encoding="utf-8").read() == "\n".join(REP), "SURVEY READ-BACK MISMATCH"
print("survey lines:", len(REP))

In [ ]:
# CELL 6: report
print("=== SSC-MEK PHASE A REPORT BEGIN ===")
print("NO SCORE AND NO DISEASE SIGNATURE WAS COMPUTED IN THIS NOTEBOOK")
print("LIBRARY", LIB)
print("--- CELL LINES (QC-pass exemplar trt_cp signatures) ---"); print(CT.to_string(index=False))
print("--- MOA VOCABULARY (MEK / ERK / MAPK) ---"); print(VOC.to_string())
print("--- COMPOUNDS ---"); print(D[["why", "pert_id", "cmap_name", "moa", "target", "inchi_key", "n_lib_sigs", "n_cells"]].to_string(index=False))
print("--- CELL LINES PER COMPOUND ---")
for _, r in D[D.n_lib_sigs > 0].iterrows(): print(f"{r.cmap_name} [{r.pert_id}] {r.cells}")
print("--- SSC SKIN SERIES (excluded by design and not fetched: GSE138669, GSE195452) ---"); print("\n".join(REP))
print("=== SSC-MEK PHASE A REPORT END ===")